# ABDS 画像ホスティング (管理者用)

カード画像をGoogleドライブにアップロードし、アプリから参照可能にします。

## 処理内容
1. 公式サイトから全カード画像をダウンロード
2. Googleドライブの共有フォルダにアップロード
3. カード番号 → ファイルID のマッピングJSONを生成
4. アプリのdata/に配置するJSONを出力

## Step 1: Googleドライブをマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: 設定 & カード番号取得

In [ ]:
import os, json, time, urllib.request
from IPython.display import clear_output

# === 設定 ===
DRIVE_FOLDER = '/content/drive/MyDrive/ABDS_CardImages'
CARD_LIST_URL = 'https://sekimiya.github.io/abds/data/card_numbers.json'
IMG_BASE = 'https://www.gundam-ab.com/images/cardlist/card/'
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Referer': 'https://www.gundam-ab.com/cardlist/'
}
DELAY = 0.05

os.makedirs(DRIVE_FOLDER, exist_ok=True)

# カード番号取得
req = urllib.request.Request(CARD_LIST_URL)
with urllib.request.urlopen(req, timeout=30) as resp:
    card_numbers = json.loads(resp.read().decode('utf-8'))

print(f'カード数: {len(card_numbers)}')
print(f'画像数(表+裏): {len(card_numbers) * 2}')
print(f'保存先: {DRIVE_FOLDER}')

## Step 3: 画像ダウンロード (公式サイト → Googleドライブ)

In [ ]:
def download_image(card_num, suffix=''):
    fname = f'{card_num}{suffix}.jpg'
    fpath = os.path.join(DRIVE_FOLDER, fname)
    if os.path.exists(fpath) and os.path.getsize(fpath) > 1000:
        return 'skip'
    url = f'{IMG_BASE}{card_num}{suffix}.jpg'
    try:
        req = urllib.request.Request(url, headers=HEADERS)
        with urllib.request.urlopen(req, timeout=15) as resp:
            data = resp.read()
            if len(data) > 1000:
                with open(fpath, 'wb') as f:
                    f.write(data)
                return 'ok'
    except:
        pass
    return 'fail'

total = len(card_numbers) * 2
done = 0; downloaded = 0; skipped = 0; failed = 0
start = time.time()

for i, num in enumerate(card_numbers):
    for suf in ['', '_b']:
        r = download_image(num, suf)
        if r == 'ok': downloaded += 1
        elif r == 'skip': skipped += 1
        else: failed += 1
        done += 1
        if DELAY > 0 and r == 'ok': time.sleep(DELAY)
    if (i+1) % 100 == 0 or i == len(card_numbers)-1:
        elapsed = time.time() - start
        remaining = (total - done) / (done / elapsed) if done > 0 else 0
        clear_output(wait=True)
        pct = done / total * 100
        print(f'[{"#"*int(pct//2)}{"-"*(50-int(pct//2))}] {pct:.1f}%')
        print(f'{done}/{total} (DL:{downloaded} skip:{skipped} fail:{failed})')
        print(f'経過: {elapsed:.0f}秒 / 残り: {remaining:.0f}秒')

print(f'\n完了! DL:{downloaded} skip:{skipped} fail:{failed}')

## Step 4: フォルダを公開共有設定にする

Google Drive APIを使ってフォルダを「リンクを知っている全員が閲覧可能」に設定します。

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build

auth.authenticate_user()
drive_service = build('drive', 'v3')

# ABDS_CardImages フォルダのIDを取得
results = drive_service.files().list(
    q="name='ABDS_CardImages' and mimeType='application/vnd.google-apps.folder' and trashed=false",
    fields='files(id, name)'
).execute()
folders = results.get('files', [])

if not folders:
    print('ERROR: ABDS_CardImages フォルダが見つかりません')
else:
    folder_id = folders[0]['id']
    print(f'フォルダID: {folder_id}')

    # 公開共有設定
    try:
        drive_service.permissions().create(
            fileId=folder_id,
            body={'type': 'anyone', 'role': 'reader'},
            fields='id'
        ).execute()
        print('フォルダを公開共有に設定しました')
    except Exception as e:
        if 'already' in str(e).lower():
            print('既に公開共有されています')
        else:
            print(f'共有設定エラー: {e}')

## Step 5: 全画像ファイルのIDを取得してマッピングJSON生成

In [ ]:
import re

# フォルダ内の全ファイルを取得
print('ファイル一覧を取得中...')
all_files = []
page_token = None
while True:
    results = drive_service.files().list(
        q=f"'{folder_id}' in parents and trashed=false",
        fields='nextPageToken, files(id, name)',
        pageSize=1000,
        pageToken=page_token
    ).execute()
    all_files.extend(results.get('files', []))
    page_token = results.get('nextPageToken')
    if not page_token:
        break
    print(f'  {len(all_files)}件取得...')

print(f'合計: {len(all_files)}ファイル')

# ファイル名 → ID のマッピング作成
# ファイル名パターン: AB01-001.jpg (表) / AB01-001_b.jpg (裏)
image_map = {}  # { "AB01-001": { "front": "fileId", "back": "fileId" } }

for f in all_files:
    name = f['name']
    fid = f['id']
    m = re.match(r'^([A-Za-z0-9\-]+(?:_p\d+)?)((?:_b)?)\.jpg$', name, re.I)
    if not m:
        continue
    card_num = m.group(1)
    side = 'back' if m.group(2) == '_b' else 'front'
    if card_num not in image_map:
        image_map[card_num] = {}
    image_map[card_num][side] = fid

print(f'カード数: {len(image_map)}')
front_count = sum(1 for v in image_map.values() if 'front' in v)
back_count = sum(1 for v in image_map.values() if 'back' in v)
print(f'表面: {front_count}, 裏面: {back_count}')

# 確認
sample = list(image_map.items())[:3]
for num, ids in sample:
    print(f'  {num}: {ids}')

## Step 6: マッピングJSONを保存

In [ ]:
# アプリ用のマッピングJSON
# コンパクトな形式: { "AB01-001": ["frontId", "backId"], ... }
compact_map = {}
for num, ids in sorted(image_map.items()):
    compact_map[num] = [ids.get('front', ''), ids.get('back', '')]

# Googleドライブに保存
output_path = '/content/drive/MyDrive/ABDS_CardImages/drive_image_map.json'
with open(output_path, 'w') as f:
    json.dump(compact_map, f, separators=(',', ':'))

size_kb = os.path.getsize(output_path) / 1024
print(f'保存先: {output_path}')
print(f'サイズ: {size_kb:.1f} KB')
print(f'カード数: {len(compact_map)}')
print()
print('このJSONをアプリの data/drive_image_map.json に配置してください。')
print('ダウンロードするには下のセルを実行してください。')

In [ ]:
# ファイルをダウンロード
from google.colab import files
files.download(output_path)

## (オプション) 新カード追加時の差分アップロード

既にアップロード済みの画像はスキップし、新しい画像のみアップロード+マッピング更新します。
Step 3 が既にスキップ機能付きなので、全セルを再実行するだけでOKです。